# 00 — The IF Prediction Contract

> The law every IF notebook obeys. This notebook defines the contract, provides the
> reusable `Contract` object, and demonstrates the ledger-integrity property tests
> that all simulation notebooks import.

## The contract

| Field | Meaning |
|---|---|
| PREDICTION | the falsifiable statement, with its equation |
| BASELINE | the null/competitor model that must be beaten (or matched) |
| DATA | pinned dataset + version, or seeded synthetic universe |
| PASS | quantitative pass criterion, decided in advance |
| FALSIFIER | the observation that kills the claim — published in `SCOREBOARD.md` §Kill log if it fires |

A notebook whose contract cell is edited after results exist is invalid (RETROFIT_FORECAST).
Contracts are frozen by git commit; the timestamp is the preregistration.

In [ ]:
from dataclasses import dataclass, asdict
import json, hashlib

@dataclass(frozen=True)
class Contract:
    notebook: str
    prediction: str
    baseline: str
    data: str
    pass_criterion: str
    falsifier: str

    def seal(self) -> str:
        """Content hash — quote this in the results cell so tampering is visible."""
        blob = json.dumps(asdict(self), sort_keys=True).encode()
        return hashlib.sha256(blob).hexdigest()[:16]

# Demonstration: the flagship notebook's contract (canonical copy lives in 04)
flagship = Contract(
    notebook="04_if_causal_work_threshold",
    prediction="Pi_A = (W_intact - W_scrambled) / C_model crosses 1 as a phase boundary: discontinuity, hysteresis, universality across >=3 unrelated rule families",
    baseline="smooth monotone scaling of viability with information capacity (agency-as-gradient null)",
    data="seeded synthetic universes only; seeds recorded in evidence/",
    pass_criterion="discontinuity + hysteresis detected at consistent Pi_A ~= 1 in >=3 rule families with non-overlapping mechanics",
    falsifier="viability scales smoothly/linearly with information capacity in all families -> agency is a gradient, not a state; kill IF-H1, keep Pi_A as measurement tool",
)
print("contract seal:", flagship.seal())

## Ledger integrity (Noether gate)

Every simulation notebook imports these property tests. A rule set that leaks
energy, silently destroys entropy, or erases information without a Landauer
debit **fails the run** — the experiment never gets to report a result.

In [ ]:
EPS = 1e-9

def assert_energy_conserved(e_initial, e_final, e_exported):
    drift = abs(e_initial - e_final - e_exported)
    assert drift < EPS, f"NOETHER GATE: energy ledger leaks by {drift}"

def assert_second_law(ds_agent, ds_environment):
    assert ds_agent + ds_environment >= -EPS, (
        f"PERPETUAL_RECHARGE: total entropy fell by {-(ds_agent + ds_environment)}"
    )

def assert_landauer_debit(bits_erased, heat_paid, kT=1.0):
    import math
    minimum = bits_erased * kT * math.log(2)
    assert heat_paid >= minimum - EPS, (
        f"LEDGER DECOUPLING: erased {bits_erased} bits paying {heat_paid} < Landauer minimum {minimum}"
    )

print("ledger property tests defined: energy, second law, Landauer coupling")

## Determinism check

Rerunning any notebook with its recorded seed must reproduce byte-identical
result arrays. Notebooks record `(seed, library versions, result hash)` into
`../evidence/` as WORM entries.